Middleware

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

Summarization Middleware

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage


agent = create_agent(
    model= "groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [6]:
config = {"configurable":{"thread_id":"test-1"}}

In [7]:
questions = [
    "What is 2+2",
    "What is 10/5",
    "what is 5^5",
    "what is the capital of India"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages:{response}")
    print(f"Message len: {len(response['messages'])}")

Messages:{'messages': [HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}, id='2fe60b72-a5c3-4b67-9897-da37fc104caf'), AIMessage(content='<think>\nOkay, so the user asked "What is 2+2". Hmm, that seems pretty straightforward, but maybe I should think through it carefully. Let me start by recalling basic arithmetic. Addition is one of the fundamental operations, right? So 2 plus 2... Well, if I have two apples and add two more apples, I have four apples. That makes sense. \n\nWait, but maybe they want a more formal approach? Let me consider the Peano axioms or something. The Peano axioms define the natural numbers and addition. According to those, 2 is the successor of 1, which is the successor of 0. So adding 2 and 2 would be like starting at 2 and moving two steps forward: 2 + 1 is 3, then 3 + 1 is 4. So that gives 4. \n\nAlternatively, using set theory, if I have two sets each with two elements, the union of those sets (assuming they\'re disjoint) would ha

Token Size

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

@tool
def search_Hotel(city:str)->str:
    """Searh Hotel - returns long response to use more tokens."""
    return f"""Hotel in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $100/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""



agent = create_agent(
    model= "groq:qwen/qwen3-32b",
    tools=[search_Hotel],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)

config = {"configurable":{"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

In [10]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke({"messages": [HumanMessage(content=f"Hotels in {city}")]},config=config)
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(response['messages'])

Paris: ~731 tokens, 6 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n<think>\nOkay, let\'s see. The user asked for "Hotels in Paris." That\'s the main query. My primary objective is to extract the most relevant context from the conversation history.\n\nFirst, the SESSION INTENT should be clear. The user wants information on hotels in Paris. So the session intent is straightforward: they\'re looking for hotel recommendations or information in Paris.\n\nNext, the SUMMARY section. The conversation history only has the user\'s message about Paris hotels. There\'s no further discussion or details provided. I need to note that the user\'s request is about hotels in Paris, but since there\'s no additional context like budget, dates, or preferences mentioned, the summary should reflect that. No decisions or rejected options here.\n\nARTIFACTS: Since the user hasn\'t provided or mentioned any files, resources, or specific data, this section is empty. I should 